In [7]:
import time
from pathlib import Path
import json
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ollama import chat

from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title


In [8]:
EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

RAG_PROMPT_TEMPLATE = """Tu es un assistant expert des règles de Blood Bowl.
Réponds à la question en te basant UNIQUEMENT sur le contexte fourni ci-dessous.
Si le contexte ne permet pas de répondre, dis-le clairement.

Contexte :
{context}

Question : {question}
"""

In [9]:
def ask_llm(message, model="llm", thinking = True):
    """
    Send one message to the LLM and return its answer.

    Arguments:
    prompt -- the message to send, as a string

    Returns:
    answer -- the text generated by the model
    """

    if model == "llm":

        response = chat(model="qwen3:1.7b", messages=message, think=thinking)
        answer = response.message.content
        thinking_text = response.message.thinking if thinking else None
        return answer, thinking_text

    else:
        response = chat(model="qwen2.5vl:3b", messages=message)
        answer = response.message.content
        thinking_text = response.message.thinking if thinking else None
        return answer, thinking_text


In [10]:
def ask_rag(vector_store, question: str, thinking: bool, top_k: int = 4) -> dict:
    """Pipeline complet : retrieval + génération."""
    relevant_chunks = vector_store.search(question, top_k=top_k)
    context = "\n\n---\n\n".join(relevant_chunks)
    prompt = RAG_PROMPT_TEMPLATE.format(context=context, question=question)

    answer, thinking = ask_llm(prompt, thinking)
    return {
        "question": question,
        "answer": answer,
        "thinking": thinking,
        "context_used": relevant_chunks,
    }

In [11]:
def partition_document(file_path: str):
    """ Extraction des éléments du pdf par unstructured """

    print(f"Extraction du document : {file_path.split("\\")[-1]}")

    elements = partition_pdf(
        filename=file_path, # Path du fichier pdf
        strategy="hi_res", # Methode plus précise d'extraction, mais plus lente
        infer_table_structure=True, # Garde les tables comme du html
        extract_image_block_types=["Image"], # Récupère les images du pdf
        extract_image_block_to_payload=True, # Conserve les images en base64
        languages=["fra", "eng"] # Spécification de la langue des pdf
    )
    print(f"Extraction de {len(elements)} éléments")
    return elements

In [12]:
path = "data/rules_bb.pdf"

elements = partition_document(path)

Extraction du document : data/rules_bb.pdf


KeyboardInterrupt: 

In [ ]:
elements[41].to_dict()

{'type': 'NarrativeText',
 'element_id': 'f7ee2868e5fd8f2bd511489264ab0694',
 'text': 'Ce gabarit est utilisé quand quelque chose Valdingue ou Dévie, comme décrit en page 23. On le place à côté du terrain de sorte que les deux joueurs puissent facilement le voir et qu’il corresponde à la grille sur le terrain. Chaque fois que quelque chose requiert le Gabarit de Direction Aléatoire, comme lors d’un Valdingue ou d’une Déviation, jetez un D8 et déplacez l’objet sur la case correspondant au jet, comme indiqué sur le schéma ci-dessous :',
 'metadata': {'detection_class_prob': 0.954463541507721,
  'is_extracted': 'true',
  'coordinates': {'points': ((np.float64(1513.043212890625),
     np.float64(1338.1160888671875)),
    (np.float64(1513.043212890625), np.float64(1946.737740597222)),
    (np.float64(2771.790972222222), np.float64(1946.737740597222)),
    (np.float64(2771.790972222222), np.float64(1338.1160888671875))),
   'system': 'PixelSpace',
   'layout_width': 2894,
   'layout_height':

In [ ]:
images = [element for element in elements if element.category == "Image"]
print(f"Found {len(images)} images")

# https://codebeautify.org/base64-to-image-converter

Found 116 images


In [ ]:
tables = [element for element in elements if element.category == "Table"]
print(f"Found {len(tables)} tables")

# https://jsfiddle.net/

Found 18 tables


In [ ]:
tables[1].to_dict()

{'type': 'Table',
 'element_id': 'c82e13d7d71bae644caad031ea98c52d',
 'text': 'ÉVÈNEMENTS DE COUPS D’ENVOI 2 À mort l’Arbitre : Chaque équipe reçoit immédiatement 1 Coup de Pouce de Pot-de-vin gratuit. Ce Pot-de-vin doit être utilisé avant la fin du match, ou bien il est perdu. 3 Temps Mort : Si le Marqueur de Tour de l’équipe qui engage est sur le 6, 7 ou 8 de la mi-temps, reculez de 1 case le Marqueur de Tour des deux équipes. Sinon, avancez de 1 case le Marqueur de Tour des deux équipes. Solide Défense : Le Coach de l’équipe qui engage choisit jusqu’à D3+3 joueurs Démarqués de son équipe. Les 4 joueurs choisis sont ensuite retirés du terrain et peuvent être à nouveau placés selon les restrictions habituelles du placement d’équipe. 5 Chandelle : 1 joueur Démarqué de l’équipe qui réceptionne peut immédiatement être placé sur la case sur laquelle le ballon va atterrir. Fans en Folie : Chaque Coach jette un D6 et y ajoute le nombre de Cheerleaders sur sa Fiche d’Équipe. La 6 première Ac

In [13]:
def create_chunks_by_title(elements):
    """ """
    print("Creation des chunks")

    chunks = chunk_by_title(
        elements,
        max_characters=3000,
        new_after_n_chars=2400,
        combine_text_under_n_chars=500
    )

    print(f"Creation de {len(chunks)} chunks")
    return chunks

In [ ]:
chunks = create_chunks_by_title(elements)

Creation des chunks
Creation de 321 chunks


In [ ]:
for chunk in chunks:
    print(chunk.metadata.orig_elements)

[<unstructured.documents.elements.Header object at 0x000002308BB7D1D0>, <unstructured.documents.elements.NarrativeText object at 0x000002308BBF9160>, <unstructured.documents.elements.NarrativeText object at 0x000002308BBF9CC0>]
[<unstructured.documents.elements.Title object at 0x000002308BBFA350>, <unstructured.documents.elements.NarrativeText object at 0x000002308BBF8AD0>, <unstructured.documents.elements.NarrativeText object at 0x000002308BBFAC80>]
[<unstructured.documents.elements.Title object at 0x000002308BBF8590>, <unstructured.documents.elements.Title object at 0x000002308BBFAD60>, <unstructured.documents.elements.NarrativeText object at 0x000002308BBF9EF0>, <unstructured.documents.elements.NarrativeText object at 0x000002308BBFA270>, <unstructured.documents.elements.NarrativeText object at 0x000002308BBF8D70>, <unstructured.documents.elements.NarrativeText object at 0x000002308BBF84B0>, <unstructured.documents.elements.NarrativeText object at 0x000002308BBFB230>, <unstructured.

In [ ]:
chunks[4].metadata.orig_elements[2].to_dict()

{'type': 'NarrativeText',
 'element_id': '5b20a357127dca21595eda8810bcbfe9',
 'text': 'Sans conteste la chose la plus important sur le terrain ; après tout, c’est ce dont les deux équipes ont besoin pour marquer ! Lorsqu’il n’est pas entre les mains d’un joueur, le ballon est placé au sol entièrement sur une seule case. Tant qu’un joueur est en possession du ballon, celui-ci sera placé sur le socle du joueur pour montrer que ce dernier le détient. Le ballon est un élément essentiel de n’importe quelle partie, et il est habituellement représenté par une figurine, comme vos joueurs. Chaque équipe de Blood Bowl est fournie avec au moins un ballon de Blood Bowl que vous pouvez utiliser en jeu.',
 'metadata': {'detection_class_prob': 0.9524504542350769,
  'is_extracted': 'true',
  'coordinates': {'points': ((np.float64(137.4730224609375),
     np.float64(969.1682961527773)),
    (np.float64(137.4730224609375), np.float64(1760.5571850416666)),
    (np.float64(1394.89453125), np.float64(1760.

In [ ]:
list_element_type = []
for chunk in chunks:
    if hasattr(chunk, 'metadata') and hasattr(chunk.metadata, 'orig_elements'):
        for element in chunk.metadata.orig_elements:
            element_type = type(element).__name__
    if element_type not in list_element_type:
        list_element_type.append(element_type)
print(list_element_type)

['NarrativeText', 'Image', 'ListItem', 'Text', 'Table', 'Title', 'FigureCaption', 'Header']


In [14]:
def separate_content_types(chunk):
    """ """
    content_data = {
        'text': chunk.text,
        'tables': [],
        'images': [],
        'types': ['text']
    }

    if hasattr(chunk, 'metadata') and hasattr(chunk.metadata, 'orig_elements'):
        for element in chunk.metadata.orig_elements:
            element_type = type(element).__name__

            # Gerer Tables
            if element_type == 'Table':
                content_data['types'].append('table')
                table_html = getattr(element.metadata, 'text_as_html', element.text)
                content_data['tables'].append(table_html)

            elif element_type == 'Image':
                if hasattr(element, 'metadata') and hasattr(element.metadata, 'image_base64'):
                    content_data['types'].append('image')
                    content_data['images'].append(element.metadata.image_base64)

    content_data['types'] = list(set(content_data['types']))

    return content_data
    
def create_ai_enhanced_summary(text, tables, images, thinking=True):

    try:
        prompt_text = f"""Vous créez une description interrogeable pour la recherche de contenu de documents.

        Contenu à analyser:
        Contenu textuel:
        {text}

        """

        if tables:
            prompt_text += "Tables:\n"
            for i, table in enumerate(tables):
                prompt_text += f"Table {i+1}:\n{table}\n\n"

        prompt_text += """ 
            Tes tâches:
            Générer une description compréhensible et consultable qui couvre:

            1. Les points clés des faits, nombres et donnnées du text et des tables
            2. Le topic principal et les concepts abordés
            3. les questions auquelles ce contenu pêut répondre
            4. analse du contenu visuel (graphiques, diagrammes, patterne dans les images)
            5. Les termes de recherche alternative que les utilisateurs pourraient utiliser

            Fait que ce soit détaillé et consultable - priorise la facilité de repérage à la brièveté

            DESCIPTION CONSULTABLE:"""
        
        message = {"role": "user", "content": prompt_text}
        if images:
            message["images"] = images
            answer, thinking = ask_llm([message], "vlm", thinking)
        else:
            answer, thinking = ask_llm([message], "llm", thinking)

        return answer, thinking
    
    except Exception as e:
        print(f"Résumé IA échoué : {e}")
        fallback = f"{text[:300]}..."

        return fallback, None

    
def summarise_chunks(chunks, thinking = True):
    """ """
    print("Transformation des chunks avec résumé par IA")

    langchain_documents = []
    total_chunks = len(chunks)

    for i, chunk in enumerate(chunks):
        current_chunk = i+1
        print(f"Chunk {current_chunk}/{total_chunks}")

        content_data = separate_content_types(chunk)

        print(f"Types trouvés : {content_data['types']}")
        print(f"Tables : {len(content_data['tables'])}, Images : {len(content_data['images'])}")

        if content_data['tables'] or content_data['images']:
            print(f"Creation d'un résumé par IA pour le contenu mixte")
            try:
                enhanced_content, think = create_ai_enhanced_summary(
                    content_data['text'],
                    content_data['tables'],
                    content_data['images'],
                    thinking
                )
                print(f"Résumé réussi")
                print(f"Prévisualisation du contenu : {enhanced_content[:200]}.")
            except Exception as e:
                print(f"Résumé échoué")
                enhanced_content = content_data['text']
                think = None

        else:
            print(f" Utilisation du texte brut")
            enhanced_content = content_data['text']
            think = None
        
        doc = Document(
            page_content= enhanced_content,
            metadata = {
                "original_content": json.dumps({
                    "raw_text": content_data['text'],
                    "tables_html": content_data['tables'],
                    "images_base64": content_data['images'],
                    "thinking": think
                })
            }
        )

        langchain_documents.append(doc)

    print(f"Réalisé {len(langchain_documents)} chunks")
    return langchain_documents

In [ ]:
processed_chunks =  summarise_chunks(chunks, thinking=False)

Transformation des chunks avec résumé par IA
Chunk 1/321
Types trouvés : ['text']
Tables : 0, Images : 0
 Utilisation du texte brut
Chunk 2/321
Types trouvés : ['text']
Tables : 0, Images : 0
 Utilisation du texte brut
Chunk 3/321
Types trouvés : ['image', 'text']
Tables : 0, Images : 2
Creation d'un résumé par IA pour le contenu mixte
Résumé réussi
Prévisualisation du contenu : ### Description Consultable du Terrain de Blood Bowl

#### Contenu Textuel
**Terrain de Blood Bowl :**
- **Zones d'En-But (1):** Ces zones se trouvent aux bords courts du terrain, où les joueurs doive.
Chunk 4/321
Types trouvés : ['text']
Tables : 0, Images : 0
 Utilisation du texte brut
Chunk 5/321
Types trouvés : ['text']
Tables : 0, Images : 0
 Utilisation du texte brut
Chunk 6/321
Types trouvés : ['text']
Tables : 0, Images : 0
 Utilisation du texte brut
Chunk 7/321
Types trouvés : ['image', 'text']
Tables : 0, Images : 1
Creation d'un résumé par IA pour le contenu mixte
Résumé réussi
Prévisualisation du co

In [15]:
def export_chunk_to_json(chunks, filename="chunks_export.json"):
    """ """
    export_data = []

    for i, doc in enumerate(chunks):
        chunk_data = {
            "chunk_id": i+1,
            "enhanced_content": doc.page_content,
            "metadata": {
                "original_content": json.loads(doc.metadata.get("original_content", "{}"))
            }
        }
        export_data.append(chunk_data)

    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(export_data, f, indent=2, ensure_ascii=False)

    print(f"{len(export_data)} chunks exportés dans le fichier {filename}")
    return export_data

In [ ]:
json_data = export_chunk_to_json(processed_chunks)

321 chunks exportés dans le fichier chunks_export.json


In [16]:
def create_vector_store(documents, persist_directory="db/chroma.db"):
    """ """
    print("Creation des embeddings et storage dans ChromaDB...")

    embedding_model = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)

    print("--- Creation du Vector Store")
    vectorstore = Chroma.from_documents(
        documents=documents,
        embedding=embedding_model,
        persist_directory=persist_directory,
        collection_metadata={"hnsw:space": "cosine"}
    )
    print("--- Création Terminée")

    print(f"Vector Store crée et sauvegardé {persist_directory}")
    return vectorstore

In [ ]:
db = create_vector_store(processed_chunks)

Creation des embeddings et storage dans ChromaDB...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

--- Creation du Vector Store
--- Création Terminée
Vector Store crée et sauvegardé db/chroma.db


In [ ]:
question = "Est-ce qu'il faut lancer la répulsion sur le deuxième blocage de frénésie?"
retriver = db.as_retriever(search_kwargs={"k": 5})
chunks = retriver.invoke(question)

export_chunk_to_json(chunks, "rag_result.json")

5 chunks exportés dans le fichier rag_result.json


[{'chunk_id': 1,
  'enhanced_content': 'JETS À RÉSULTAT REQUIS\n\n• On doit toujours accepter le second résultat, même s’il est pire que le premier ; on ne peut jamais relancer une relance.\n\n• Si une règle vous permet de relancer plusieurs dés, alors vous devez tous les relancer ensemble. Vous ne pouvez pas en relancer certains, attendre le résultat, puis décider si vous voulez relancer les autres.\n\nSouvent, lorsqu’on fait un jet de dé, les règles demandent d’obtenir un certain nombre suivi d’un +. Il s’agit d’un résultat requis, qui indique le score minimum nécessaire pour réussir dans une situation donnée. Par exemple, si une règle vous demande d’obtenir un 3+, alors tout jet de 3, 4, 5 ou 6 sera une réussite.',
  'metadata': {'original_content': {'raw_text': 'JETS À RÉSULTAT REQUIS\n\n• On doit toujours accepter le second résultat, même s’il est pire que le premier ; on ne peut jamais relancer une relance.\n\n• Si une règle vous permet de relancer plusieurs dés, alors vous devez

In [17]:
def run_complete_ingestion_pipeline(pdf_path: str):
    """ """

    print("Début du pipeline d'ingestion du RAG")
    print("=" * 50)

    elements = partition_document(pdf_path)

    chunks = create_chunks_by_title(elements)

    summarised_chunks = summarise_chunks(chunks)

    db=create_vector_store(summarised_chunks, persist_directory="dbv2/chroma.db")

    print("Pipeline réalisé avec succès!")
    return db

In [18]:
PDF_PATH = Path("data/")

for file in PDF_PATH.glob("*.pdf"):
    db = run_complete_ingestion_pipeline(str(file))

Début du pipeline d'ingestion du RAG
Extraction du document : Compétences-FR-EN.pdf


Loading weights:   0%|          | 0/367 [00:00<?, ?it/s]

Extraction de 45 éléments
Creation des chunks
Creation de 9 chunks
Transformation des chunks avec résumé par IA
Chunk 1/9
Types trouvés : ['text']
Tables : 0, Images : 0
 Utilisation du texte brut
Chunk 2/9
Types trouvés : ['table', 'text']
Tables : 1, Images : 0
Creation d'un résumé par IA pour le contenu mixte
Résumé réussi
Prévisualisation du contenu : ### **Résumé des éléments clés :**

1. **Nombre de termes :**  
   Le texte liste **34 termes** liés à des concepts en football (techniques, stratégies, mouvements, etc.), organisés dans des catégorie.
Chunk 3/9
Types trouvés : ['text']
Tables : 0, Images : 0
 Utilisation du texte brut
Chunk 4/9
Types trouvés : ['table', 'text']
Tables : 1, Images : 0
Creation d'un résumé par IA pour le contenu mixte
Résumé réussi
Prévisualisation du contenu : ### **Description Consultable**  
#### **1. Points clés des faits, nombres et données**  
- **Termes principaux** :  
  - **Catégories** : "Always Hungry," "Animosity," "Animal Savagery," "Ball 

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

--- Creation du Vector Store
--- Création Terminée
Vector Store crée et sauvegardé dbv2/chroma.db
Pipeline réalisé avec succès!
Début du pipeline d'ingestion du RAG
Extraction du document : fre_20-05_blood_bowl_faq_errata.pdf
Extraction de 182 éléments
Creation des chunks
Creation de 13 chunks
Transformation des chunks avec résumé par IA
Chunk 1/13
Types trouvés : ['image', 'text']
Tables : 0, Images : 18
Creation d'un résumé par IA pour le contenu mixte
Résumé IA échoué : {"error":{"code":400,"message":"request (12007 tokens) exceeds the available context size (4096 tokens), try increasing it","type":"exceed_context_size_error","n_prompt_tokens":12007,"n_ctx":4096}} (status code: 400)
Résumé réussi
Prévisualisation du contenu : MARI AMIMIER:

BAPOUN SOUL,

ef

FUUTSALL

Le JEU DE

FANTASTIQUE:

COMMENTAIRES DE CONCEPTION MAI 2026

L es errata suivants corrigent les erreurs du Livre de Règles de Blood Bowl, de tous les numéro.
Chunk 2/13
Types trouvés : ['text']
Tables : 0, Images : 0


...

--- Creation du Vector Store
--- Création Terminée
Vector Store crée et sauvegardé dbv2/chroma.db
Pipeline réalisé avec succès!
Début du pipeline d'ingestion du RAG
Extraction du document : starplayer_bb.pdf
Extraction de 243 éléments
Creation des chunks
Creation de 19 chunks
Transformation des chunks avec résumé par IA
Chunk 1/19
Types trouvés : ['text']
Tables : 0, Images : 0
 Utilisation du texte brut
Chunk 2/19
Types trouvés : ['image', 'text']
Tables : 0, Images : 1
Creation d'un résumé par IA pour le contenu mixte
Résumé IA échoué : {"error":{"code":400,"message":"request (4963 tokens) exceeds the available context size (4096 tokens), try increasing it","type":"exceed_context_size_error","n_prompt_tokens":4963,"n_ctx":4096}} (status code: 400)
Résumé réussi
Prévisualisation du contenu : LISTE DES STAR PLAYERS !

L es pages suivantes vous présentent certains des Star Players actuellement disponibles pour Blood Bowl. Chaque joueur s’est taillé une solide réputation dans les annales

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

--- Creation du Vector Store
--- Création Terminée
Vector Store crée et sauvegardé dbv2/chroma.db
Pipeline réalisé avec succès!
Début du pipeline d'ingestion du RAG
Extraction du document : teams_bb.pdf
Extraction de 278 éléments
Creation des chunks
Creation de 61 chunks
Transformation des chunks avec résumé par IA
Chunk 1/61
Types trouvés : ['text']
Tables : 0, Images : 0
 Utilisation du texte brut
Chunk 2/61
Types trouvés : ['table', 'text']
Tables : 1, Images : 0
Creation d'un résumé par IA pour le contenu mixte
Résumé réussi
Prévisualisation du contenu : ### **Description Consultable**  

---

#### **1. Points Clés des Faits, Nombres et Données**  
- **Rôles et Coûts** :  
  - Le **Blitzer Nain** est le rôle le plus coûteux (100k), suivit par le **Gro.
Chunk 3/61
Types trouvés : ['text']
Tables : 0, Images : 0
 Utilisation du texte brut
Chunk 4/61
Types trouvés : ['table', 'text']
Tables : 1, Images : 0
Creation d'un résumé par IA pour le contenu mixte
Résumé réussi
Prévisualisatio

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

--- Creation du Vector Store
--- Création Terminée
Vector Store crée et sauvegardé dbv2/chroma.db
Pipeline réalisé avec succès!


In [ ]:
question = "Est-ce qu'il faut lancer la répulsion sur le deuxième blocage de frénésie?"

In [28]:
def generate_final_answer(question, thinking = True):
    """ """
    try:
        retriver = db.as_retriever(search_kwargs={"k": 3})
        chunks = retriver.invoke(question)

        prompt_text = f"""A partir des documents suivants, réponse à cette question:
        {question}

        Contenu à analyser:
        """
        message = {"role": "user", "images": []}

        for i, chunk in enumerate(chunks):
            prompt_text += f"--- Document {i+1} ---\n"
            if "original_content" in chunk.metadata:
                original_data = json.loads(chunk.metadata['original_content'])

                raw_text = original_data.get("raw_text", "")
                if raw_text:
                    prompt_text += f"Texte:\n{raw_text}\n\n"

                tables_html = original_data.get("tables_html", [])
                if tables_html:
                    prompt_text += "Tableau:\n"

                    for j, table in enumerate(tables_html):
                        prompt_text += f"Tableau {j+1}:\n{table}\n\n"

                images = original_data.get("images_base64", [])
                if images:
                    for image in images:
                        message["images"].append(image)
                    is_image = True
                else:
                    is_image = False

            prompt_text += "\n"

        prompt_text += """
        faire une réponse claire et compréhensive en utilisant le texte, les tableaux et les images ci-dessus. Si les documents ne permettent pas de répondre à la question, dis-le clairement.
        """
        message["content"] = prompt_text

        if is_image:
            answer, think = ask_llm([message], "vlm", thinking)
        else:
            answer, think = ask_llm([message], 'llm', thinking)
        
        return answer, think
    except Exception as e:
        print(f"Génération de la réponse échouée: {e}")
        return None, None

In [25]:
answer, think = generate_final_answer(chunks, question)

In [26]:
print(answer)

**Réponse :**  
Les documents analysés ne fournissent pas d'information claire ou spécifique concernant la nécessité de lancer la répulsion sur le deuxième blocage de frénésie. Les textes mentionnent des règles relatives à l'agression, aux blocages, et aux jets d'Appuis Sûrs, mais **aucun élément ne permet de déterminer si la répulsion doit être lancée en cas de blocage secondaire**. 

- **Contexte des documents** :  
  - Le document 1 évoque des règles de blocage (Bousculé, Défenseur Plaquée, etc.), mais **pas de directive spécifique sur la répulsion**.  
  - Les règles sur l'Appuis Sûrs (qui dépendent de la condition de Plaqué ou Chute) ne sont pas liées à la répulsion.  
  - Aucun des documents ne mentionne explicitement une règle sur la répulsion en réponse à un blocage secondaire.  

**Conclusion** :  
Les documents ne permettent pas de répondre à la question. Il n'y a **aucune indication** dans les textes fournis que la répulsion soit nécessaire pour un blocage secondaire de frén

In [27]:
print(think)

Okay, let's tackle this question. The user is asking if they need to launch repulsion against the second blockage in the frenzy. The documents provided are in French, so I need to parse them carefully.

First, looking at the content of the documents. There are three documents. The first two are about gameplay mechanics, specifically regarding aggro, blockage, and appuis sûrs. The third document talks about required results for jets.

The user's question is about whether repulsion is needed for the second blockage. The answer needs to be based on the documents. Let me check the documents again.

In Document 1, there's a section about blockage. The key points are: when you're targeted by a blockage and get a result like Bousculé or Défenseur Plaquée, and getting a 1 when you run. Also, the rules for aggression (like Sournois or Agresseur Solitaire) apply to aggression granted by the Marteau-pilon competence. The answer to the question about free aggression and additional actions is "Oui 

In [29]:
question = "Quels sont les faces du dé de blocage"

answer, think = generate_final_answer(question)

print(answer)

Génération de la réponse échouée: {"error":{"code":400,"message":"Multimodal data provided, but model does not support multimodal requests.","type":"invalid_request_error"}} (status code: 400)
None
